# Data Description
- Author: 待學生填寫（已詢問）
- Date first created: 原資料建立日期未提供；本說明建立於 2026-11-07
- Background: 使用者指定的 MA／MS 房貸申請整理檔；用於依州、種族與收入比較核准率。
- Unit of observation: 每列視為一筆申請紀錄；原始申請識別碼未包含於此整理檔，無法驗證逐筆唯一性。
- Sample period: 未提供；本檔無年份欄位，不因其他 ACS 檔案為 2024 年就推定本檔年份。
- Known issues: 收入缺失 14,620 筆、負收入 189 筆；15 欄完全相同的重複列 982 筆，沒有識別碼時不可判定為重複申請，不刪除。原始來源與上游處理程式未提供。
- Definition for each variable:
    - 分析使用：state_code 州別；derived_race 彙整種族；income 信貸決策使用的年收入（HMDA 定義為千美元）；action_taken 申請結果。使用者指定 1、2 為核准，3 為拒絕。
    - 其餘原始欄位：county_code 郡代碼、derived_ethnicity 彙整族裔、derived_sex 彙整性別、loan_type 貸款類型、loan_purpose 貸款用途、loan_amount 貸款金額、loan_to_value_ratio 貸款價值比、property_value 房產價值、occupancy_type 使用類型、debt_to_income_ratio 債務收入比、applicant_age 年齡組。這些欄位不參與本次分析，未驗證其上游轉換定義。
- Data path: [state_MA-MS_income_15_columns.csv](state_MA-MS_income_15_columns.csv)
- Source data: 使用者指定的外部已整理檔；下載來源與資料年份待提供。
- Processing code: not available（未提供，不重建上游處理）
- Role: analysis
- Verification status: 已核對 237,047 列、15 欄、州別 MA／MS、結果碼 1／2／3、分析必要欄位及缺失數；不表示已驗證抽樣代表性或原始申請唯一性。
- SHA-256: `f466ff6c5ee8694b6a45e5d493dc38401cb75bc745131a6cb2f2b294190a4369`
- Downstream analysis input: `data/processed/state_MA-MS_income_15_columns.csv`

欄位對照參考：[HMDA 官方 Public LAR Data Fields](https://ffiec.cfpb.gov/documentation/publications/loan-level-datasets/lar-data-fields/)。欄位定義對照不證明本檔的下載來源或年份。


In [1]:
from pathlib import Path
import hashlib
import pandas as pd
from IPython.display import display

start = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (start, *start.parents) if (p / 'AGENTS.md').is_file() and (p / 'templates').is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('請從 econ-5166-student-kit 或其子目錄執行。')


## Change for Your Own Data
固定精確輸入路徑與已核對 SHA-256；不變動原資料。

In [2]:
INPUT_PATH = 'data/processed/state_MA-MS_income_15_columns.csv'
EXPECTED_SHA256 = 'f466ff6c5ee8694b6a45e5d493dc38401cb75bc745131a6cb2f2b294190a4369'
input_file = (PROJECT_ROOT / INPUT_PATH).resolve()
assert input_file.is_relative_to((PROJECT_ROOT / 'data/processed').resolve())
if hashlib.sha256(input_file.read_bytes()).hexdigest() != EXPECTED_SHA256:
    raise ValueError('資料版本已改變，請先核對資料來源。')
df = pd.read_csv(input_file)


## Summary

In [3]:
print({'rows':len(df), 'columns':len(df.columns), 'sha256':EXPECTED_SHA256})
display(df.dtypes.to_frame('資料型態'))

{'rows': 237047, 'columns': 15, 'sha256': 'f466ff6c5ee8694b6a45e5d493dc38401cb75bc745131a6cb2f2b294190a4369'}


,資料型態
state_code,str
county_code,float64
derived_ethnicity,str
derived_race,str
derived_sex,str
action_taken,int64
loan_type,int64
loan_purpose,int64
loan_amount,float64
loan_to_value_ratio,str


### Sample

In [4]:
display(df.head(10))

,state_code,county_code,derived_ethnicity,derived_race,derived_sex,action_taken,loan_type,loan_purpose,loan_amount,loan_to_value_ratio,property_value,occupancy_type,income,debt_to_income_ratio,applicant_age
0,MA,25023.0,Not Hispanic or Latino,Black or African American,Male,1,1,1,605000.0,95.00000,635000,1,147.0,45,25-34
1,MS,28071.0,Not Hispanic or Latino,White,Female,1,2,32,125000.0,57.80000,205000,1,44.0,20%-<30%,65-74
2,MS,28059.0,Ethnicity Not Available,Race Not Available,Sex Not Available,1,3,1,185000.0,99.99900,185000,1,83.0,40,35-44
3,MA,25021.0,Not Hispanic or Latino,Black or African American,Female,3,2,32,245000.0,77.42000,315000,1,73.0,>60%,35-44
4,MS,28049.0,Not Hispanic or Latino,White,Joint,1,2,1,335000.0,89.62600,365000,1,94.0,30%-<36%,35-44
5,MS,28035.0,Not Hispanic or Latino,White,Male,1,3,1,295000.0,100.00000,285000,1,74.0,50%-60%,65-74
6,MS,28049.0,Not Hispanic or Latino,White,Male,1,2,32,295000.0,80.00000,355000,1,110.0,38,25-34
7,MA,25027.0,Not Hispanic or Latino,White,Joint,1,3,31,395000.0,NaN,NaN,1,NaN,NaN,25-34
8,MA,25023.0,Not Hispanic or Latino,White,Joint,2,2,31,485000.0,NaN,NaN,1,NaN,NaN,45-54
9,MS,28047.0,Not Hispanic or Latino,White,Joint,1,3,31,315000.0,NaN,NaN,1,NaN,NaN,35-44


### Summary Stats
數值摘要 N 為非缺失筆數；類別比例以該欄非缺失數為分母。

In [5]:
display(df[['income']].describe().T)
for col in ['state_code','derived_race','action_taken']:
    print(col, '缺失數', int(df[col].isna().sum()))
    counts=df[col].value_counts(dropna=True)
    display(pd.DataFrame({'筆數': counts, '比例 (%)': counts / counts.sum()*100}))
print('收入缺失',df.income.isna().sum(), '負收入', (df.income<0).sum(), '完全相同列',df.duplicated().sum())

,count,mean,std,min,25%,50%,75%,max
income,222427.0,161.255877,486.291362,-5536.0,72.0,114.0,182.0,150000.0


state_code 缺失數 0


,筆數,比例 (%)
state_code,,
MA,159998,67.496319
MS,77049,32.503681


derived_race 缺失數 0


,筆數,比例 (%)
derived_race,,
White,149260,62.966416
Race Not Available,37554,15.842428
Black or African American,31370,13.233663
Asian,12773,5.388383
Joint,4313,1.819470
American Indian or Alaska Native,850,0.358579
2 or more minority races,448,0.188992
Native Hawaiian or Other Pacific Islander,394,0.166212
Free Form Text Only,85,0.035858


action_taken 缺失數 0


,筆數,比例 (%)
action_taken,,
1,169296,71.418748
3,58993,24.886626
2,8758,3.694626


收入缺失 14620 負收入 189 完全相同列 982


### Assertions

In [6]:
assert df.shape == (237047,15)
assert {'state_code','derived_race','income','action_taken'} <= set(df.columns)
assert set(df.state_code)=={'MA','MS'}
assert set(df.action_taken)=={1,2,3}
assert df.income.isna().sum()==14620
assert hashlib.sha256(input_file.read_bytes()).hexdigest()==EXPECTED_SHA256
print('通過：輸入版本、筆數、必要欄位、州別、結果碼、收入缺失數。原始識別碼與年份未提供，未驗證。')

通過：輸入版本、筆數、必要欄位、州別、結果碼、收入缺失數。原始識別碼與年份未提供，未驗證。
